# Daily Challenge: Building Trustworthy Insights with BERT

**Scenario:** A dry run for deploying a *trustworthy* AI assistant inside your company. We inspect
BERT's attention, fine-tune a lightweight encoder, and ship an inference helper that returns both a
prediction **and** the evidence tokens behind it.

**Pipeline**
1. Data loading & inspection (`tweet_eval`, sentiment config)
2. Tokenization pipeline (`distilbert-base-uncased`, 128 tokens)
3. Fine-tuning DistilBERT in TensorFlow/Keras
4. Evaluation & calibration (accuracy, macro-F1, confidence histogram)
5. Attention inspection (`[CLS]` -> token attention heatmap)

**Deliverable:** a saved model directory + an `analyze_text()` helper returning
`{label, confidence, highlighted_tokens}`.

**Environment:** Python 3.9+, ideally a GPU runtime (Colab T4 is enough). CPU-only works but is slow.

> This notebook follows the same TensorFlow + legacy-Keras-2 setup as the Week 6 mini-project. The
> challenge text mentions the `Trainer` API (PyTorch); the equivalent in Keras is `model.compile` /
> `model.fit`, which is what we use here.

## Setup

In [ ]:
# ==========================================================================
#  STEP 1 of 2: run this install cell ONCE, then RESTART the runtime.
# ==========================================================================
# Why these exact pins (same rationale as the Week 6 mini-project):
#  * tensorflow and tf-keras MUST share the same minor version (2.20 <-> 2.20).
#    tf-keras 2.20 requires tensorflow < 2.21, so we hold tensorflow at 2.20.
#  * transformers MUST be >= 4.38 to honor TF_USE_LEGACY_KERAS. Older versions
#    ignore it and crash with: module 'keras.backend' has no attribute 'set_value'
#  * datasets is needed for tweet_eval (an HF dataset, not a tfds one).
!pip install -q \
    "tensorflow==2.20.*" \
    "tf-keras==2.20.*" \
    "transformers==4.44.2" \
    "tokenizers>=0.19,<0.20" \
    "datasets>=2.19" scikit-learn accelerate evaluate

from importlib.metadata import version, PackageNotFoundError
print("\n--- installed versions ---")
for pkg in ["tensorflow", "tf-keras", "transformers", "tokenizers", "datasets"]:
    try:
        print(f"{pkg:13s}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:13s}: NOT INSTALLED")
print("\nNow: Runtime > Restart session, then run the guard cell (Step 2) first.")

In [ ]:
# ==========================================================================
#  STEP 2 of 2: RUN THIS CELL FIRST after the restart, before everything else.
#  Fixes: module 'keras.backend' has no attribute 'set_value'
# ==========================================================================
import os, sys

os.environ["TF_USE_LEGACY_KERAS"] = "1"

# The switch above only works in a fresh kernel. If TF/Keras were already
# imported this session, stop loudly.
if "tensorflow" in sys.modules or "keras" in sys.modules:
    raise RuntimeError(
        "TensorFlow/Keras were already imported in this session, so the "
        "legacy-Keras switch cannot take effect.\n"
        "FIX: Runtime > Restart session, then run THIS cell FIRST."
    )

try:
    import tf_keras  # noqa: F401
except ImportError:
    raise RuntimeError(
        "tf-keras is not installed. Run the install cell (Step 1), restart, "
        "then run THIS cell first."
    )

import tensorflow as tf
import transformers
from packaging.version import parse

print("TensorFlow  :", tf.__version__)
print("tf.keras    :", tf.keras.__version__)
print("transformers:", transformers.__version__)

if not tf.keras.__version__.startswith("2"):
    raise RuntimeError(
        f"tf.keras is still Keras {tf.keras.__version__} (Keras 3). Ensure "
        "tensorflow and tf-keras share minor 2.20, restart, run this cell first."
    )
if parse(transformers.__version__) < parse("4.38"):
    raise RuntimeError(
        f"transformers {transformers.__version__} is too old to honor "
        "TF_USE_LEGACY_KERAS (need >= 4.38)."
    )

print("\nOK - legacy Keras 2 backend active. Safe to load DistilBERT.")

## Imports & Hardware Check

If `GPU devices detected: []` appears, switch the runtime to a GPU accelerator
(Colab: Runtime → Change runtime type → GPU).

In [ ]:
import platform
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    TFAutoModelForSequenceClassification,
    TFAutoModel,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score

tf.random.set_seed(42)
np.random.seed(42)

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

## 1. Load the tweet_eval (sentiment) Dataset

`tweet_eval` / `sentiment` ships three splits (train / validation / test) and three classes.
Unlike IMDB (which we loaded via `tfds`), this is a Hugging Face `datasets` dataset, so we load it
with `load_dataset` and convert to `tf.data` later.

In [ ]:
raw = load_dataset("tweet_eval", "sentiment")
print(raw)

# Integer labels -> names, defined by the dataset card.
LABELS = raw["train"].features["label"].names   # ['negative', 'neutral', 'positive']
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}
print("\nLabels:", id2label)
assert len(LABELS) == 3, "Expected 3 sentiment classes"


In [ ]:
# Class distribution per split
from collections import Counter

for split in ["train", "validation", "test"]:
    counts = Counter(raw[split]["label"])
    dist = {id2label[k]: counts[k] for k in sorted(counts)}
    print(f"{split:<11} (n={len(raw[split]):>6}): {dist}")


In [ ]:
# Save two example tweets per label for later visualization / attention inspection.
examples_by_label = {name: [] for name in LABELS}
for text, lab in zip(raw["train"]["text"], raw["train"]["label"]):
    name = id2label[lab]
    if len(examples_by_label[name]) < 2:
        examples_by_label[name].append(text)
    if all(len(v) == 2 for v in examples_by_label.values()):
        break

for name, exs in examples_by_label.items():
    print(f"\n=== {name.upper()} ===")
    for e in exs:
        print(" -", e)


## 2. Tokenizer Setup & Data Pipeline

`distilbert-base-uncased` uses WordPiece tokenization, adds `[CLS]`/`[SEP]`, and returns an
attention mask. Note DistilBERT has **no** `token_type_ids` (unlike base BERT). We truncate/pad to
128 tokens, then turn the tokenized HF dataset into a `tf.data.Dataset` with `to_tf_dataset`.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", tokenizer.name_or_path)

def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = batch["label"]
    return enc

tokenized = raw.map(preprocess, batched=True, remove_columns=raw["train"].column_names)
print(tokenized)


In [ ]:
# Dynamic padding per batch (pads to the batch's longest sequence, not a fixed 128).
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

def to_tf(split, shuffle):
    return tokenized[split].to_tf_dataset(
        columns=["input_ids", "attention_mask"],
        label_cols=["labels"],
        shuffle=shuffle,
        batch_size=BATCH_SIZE,
        collate_fn=collator,
    )

train_ds = to_tf("train", shuffle=True)
val_ds   = to_tf("validation", shuffle=False)
test_ds  = to_tf("test", shuffle=False)

# Peek at one batch
xb, yb = next(iter(train_ds))
print("Batch keys:", list(xb.keys()))
print("input_ids shape:", xb["input_ids"].shape, "| labels shape:", yb.shape)


## 3. Initialize and Fine-Tune the Model

`TFAutoModelForSequenceClassification` bundles the pretrained DistilBERT encoder with a fresh
3-class head. The challenge's `TrainingArguments` map to Keras like this:

| Trainer arg | Keras equivalent |
|---|---|
| `num_train_epochs=3` | `epochs=3` in `fit` |
| `per_device_train_batch_size=32` | `BATCH_SIZE=32` in the data pipeline |
| `learning_rate=5e-5` | `AdamW(learning_rate=5e-5)` |
| `weight_decay=0.01` | `AdamW(weight_decay=0.01)` |
| `load_best_model_at_end=True` (by macro-F1) | `ModelCheckpoint(save_best_only=True)` |

In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

optimizer = tf.keras.optimizers.AdamW(learning_rate=5e-5, weight_decay=0.01, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()


In [ ]:
# Keras doesn't compute macro-F1 mid-epoch, so we log it per epoch with a callback
# (the TF equivalent of Trainer's compute_metrics) and keep the best weights by val macro-F1.
class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, val_dataset, val_labels):
        super().__init__()
        self.val_dataset = val_dataset
        self.val_labels = val_labels
        self.best_f1 = -1.0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        logits = self.model.predict(self.val_dataset, verbose=0).logits
        preds = logits.argmax(axis=-1)
        f1 = f1_score(self.val_labels, preds, average="macro")
        acc = accuracy_score(self.val_labels, preds)
        print(f"  -> val_accuracy={acc:.4f}  val_macro_f1={f1:.4f}")
        if f1 > self.best_f1:
            self.best_f1 = f1
            self.best_weights = self.model.get_weights()  # restore-best-model

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)
            print(f"Restored best weights (val_macro_f1={self.best_f1:.4f}).")

val_labels = np.array(tokenized["validation"]["labels"])
f1_cb = MacroF1Callback(val_ds, val_labels)


In [ ]:
EPOCHS = 3

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[f1_cb],
)


In [ ]:
# Save the best model + tokenizer for teammates to reuse.
SAVE_DIR = "distilbert-tweeteval-sentiment"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)


In [ ]:
# Optional: plot the learning curves
hist = history.history
plt.figure(figsize=(6, 4))
plt.plot(hist["accuracy"], marker="o", label="train accuracy")
plt.plot(hist["val_accuracy"], marker="o", label="val accuracy")
plt.title("Fine-tuning accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Evaluation & Calibration

A small helper runs predictions on a split and returns logits/labels, then we report accuracy +
macro-F1 (the `compute_metrics` equivalent) and a confidence histogram.

In [ ]:
def predict_split(dataset, labels):
    logits = model.predict(dataset, verbose=0).logits
    preds = logits.argmax(axis=-1)
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"probs": probs, "preds": preds, "labels": labels, "accuracy": acc, "f1_macro": f1}

val_res = predict_split(val_ds, val_labels)
print(f"Validation accuracy : {val_res['accuracy']:.4f}")
print(f"Validation macro-F1 : {val_res['f1_macro']:.4f}")


In [ ]:
test_labels = np.array(tokenized["test"]["labels"])
test_res = predict_split(test_ds, test_labels)

test_conf = test_res["probs"].max(axis=-1)   # softmax confidence of the predicted class
print(f"Test accuracy  : {test_res['accuracy']:.4f}")
print(f"Test macro-F1  : {test_res['f1_macro']:.4f}")
print(f"Mean confidence: {test_conf.mean():.4f}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(test_conf, bins=np.arange(0.0, 1.01, 0.1), edgecolor="black", alpha=0.8)
plt.axvline(test_res["accuracy"], color="red", linestyle="--",
            label=f"test accuracy = {test_res['accuracy']:.2f}")
plt.title("Confidence histogram (softmax of predicted class)")
plt.xlabel("Predicted-class confidence")
plt.ylabel("Number of test tweets")
plt.legend()
plt.tight_layout()
plt.show()


**Calibration comment.** Compare the mass of the histogram to the red accuracy line. If most
predictions pile into the 0.9-1.0 bin while accuracy sits lower, the model is **over-confident**
(typical of cross-entropy-trained transformers) — softmax scores shouldn't be read as literal
probabilities without temperature scaling. Weight spread toward the 0.4-0.6 bins on a 3-class task
signals **genuine uncertainty**, usually on the ambiguous *neutral* class. Fill in the specific
trend you observe above, and recall the mini-project rule: only auto-act above a confidence
threshold (e.g. 0.90), route the rest to a human.

## 5. Attention Inspection

We pass one saved example through the **encoder** (`TFAutoModel`, with `output_attentions=True`),
average the last layer's heads, and read off how much attention the `[CLS]` token sends to every
other token — a lightweight "what did the model look at" explanation.

In [ ]:
encoder = TFAutoModel.from_pretrained(SAVE_DIR, output_attentions=True)

def cls_attention(text):
    enc = tokenizer(text, return_tensors="tf", truncation=True, max_length=MAX_LENGTH)
    out = encoder(**enc)
    # attentions: tuple(num_layers) of [batch, heads, seq, seq]
    last = out.attentions[-1][0]                       # [heads, seq, seq]
    avg = tf.reduce_mean(last, axis=0).numpy()         # average over heads -> [seq, seq]
    cls_to_tokens = avg[0]                             # row 0 = attention FROM [CLS]
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0].numpy())
    return tokens, cls_to_tokens

example = examples_by_label["negative"][0]
print("Tweet:", example)
tokens, weights = cls_attention(example)


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(range(len(tokens)), weights, color="steelblue")
plt.xticks(range(len(tokens)), tokens, rotation=60, ha="right")
plt.title("Attention from [CLS] to each token (last layer, head-averaged)")
plt.ylabel("Attention weight")
plt.tight_layout()
plt.show()

# Top contributing tokens (ignoring special tokens)
special = set(tokenizer.all_special_tokens)
ranked = sorted(
    [(t, float(w)) for t, w in zip(tokens, weights) if t not in special],
    key=lambda x: x[1], reverse=True,
)
print("Top tokens by [CLS] attention:")
for t, w in ranked[:8]:
    print(f"  {t:<15} {w:.3f}")


**Insight (example).** The `[CLS]` token concentrates attention on the sentiment-bearing words
(e.g. *terrible*, *love*, *worst*) rather than on function words — evidence the model bases its
prediction on the right cues. Replace this with the concrete tokens you see highlighted above.

## Deliverable: `analyze_text()`

Production-style helper returning `{label, confidence, highlighted_tokens}` — drop-in evidence for
support tooling. `highlighted_tokens` are the top tokens by `[CLS]` attention, i.e. *why* the model
chose the label.

In [ ]:
clf = TFAutoModelForSequenceClassification.from_pretrained(SAVE_DIR, output_attentions=True)

def analyze_text(text, top_k=5):
    enc = tokenizer(text, return_tensors="tf", truncation=True, max_length=MAX_LENGTH)
    out = clf(**enc)

    probs = tf.nn.softmax(out.logits, axis=-1)[0].numpy()
    pred_id = int(probs.argmax())

    # head-averaged last-layer [CLS] attention -> evidence tokens
    cls_attn = tf.reduce_mean(out.attentions[-1][0], axis=0).numpy()[0]
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0].numpy())
    special = set(tokenizer.all_special_tokens)
    ranked = sorted(
        [(t, float(w)) for t, w in zip(tokens, cls_attn) if t not in special],
        key=lambda x: x[1], reverse=True,
    )

    return {
        "label": clf.config.id2label[pred_id],
        "confidence": round(float(probs[pred_id]), 4),
        "highlighted_tokens": [t for t, _ in ranked[:top_k]],
    }

for t in [
    "The customer support was absolutely terrible and slow.",
    "Package arrived on time, nothing special.",
    "I love this product, best purchase ever!",
]:
    print(t)
    print("  ->", analyze_text(t), "\n")


### Packaging for teammates

`SAVE_DIR` now holds the TF model weights, config (with `id2label`), and tokenizer files. A teammate
reloads everything with:

```python
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
mdl = TFAutoModelForSequenceClassification.from_pretrained("distilbert-tweeteval-sentiment")
tok = AutoTokenizer.from_pretrained("distilbert-tweeteval-sentiment")
```

Zip and share `distilbert-tweeteval-sentiment/` (or push to the Hub via `model.push_to_hub()`)
alongside this notebook's attention/confidence visualizations.

## Reflection & Next Steps

**Why this matters:** we reused a public DistilBERT checkpoint to get a usable 3-class sentiment
signal with minimal task-specific training, and — crucially — made it *explainable* by surfacing the
attention-weighted evidence tokens behind each prediction.

**1. Is the model well calibrated?** Read the confidence histogram against the accuracy line. If the
mass sits far right of accuracy, apply temperature scaling before treating softmax scores as
probabilities.

**2. Where would you add guardrails?**
- **Confidence thresholding** — only auto-act above ~0.90; route the rest to a human.
- **Class-aware thresholds** — favor recall on *negative* so angry customers are never missed.
- **Out-of-domain detection** — the model learned from tweets; flag inputs that look unlike your
  support text and monitor for drift.
- **Human-in-the-loop** — never auto-close/auto-refund on sentiment alone.
- **Logging** — store prediction + confidence + highlighted tokens for audit.

**3. What's next?** domain adaptation on your own messages, multilingual checkpoints
(DistilBERT-multilingual, XLM-R), and a small dashboard tracking accuracy/drift over time.